# 2.1 · JSON-RPC 消息、_meta 与错误码

**JSON-RPC messages, _meta and error codes**

> **学完本课你能：** 能够手工读写任何 MCP 消息，正确使用 `_meta` 信封及其命名规则，并能根据 JSON-RPC 错误码诊断故障。

官方阅读材料：[Spec: Base protocol overview](https://modelcontextprotocol.io/specification/2026-07-28/basic/index) · [Schema reference](https://modelcontextprotocol.io/specification/2026-07-28/schema)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 一个值得信赖的消息分类器
在和服务器对话之前，先把本课的规则写成代码。`inspect_message` 会对一个解码后的 JSON 对象进行分类，并列出它违反的每一条规则。我们会在服务器的真实回复上复用它。

In [ ]:
import json


def show(obj, limit=900):
    text = json.dumps(obj, indent=2, ensure_ascii=False)
    print(text if len(text) <= limit else text[:limit] + "\n  ...")


REQUIRED_ENVELOPE = ("io.modelcontextprotocol/protocolVersion", "io.modelcontextprotocol/clientCapabilities")


def inspect_message(msg: dict) -> tuple[str, list[str]]:
    """Return (kind, problems) for one decoded MCP / JSON-RPC message."""
    problems = []
    if msg.get("jsonrpc") != "2.0":
        problems.append('jsonrpc must be exactly "2.0"')
    has_id = "id" in msg
    if "method" in msg:
        kind = "request" if has_id else "notification"
        if has_id:
            rid = msg["id"]
            if rid is None:
                problems.append("request id MUST NOT be null (MCP rule, stricter than JSON-RPC)")
            elif isinstance(rid, bool) or not isinstance(rid, (str, int)):
                problems.append("request id must be a string or an integer")
            meta = (msg.get("params") or {}).get("_meta") or {}
            missing = [k for k in REQUIRED_ENVELOPE if k not in meta]
            if missing:
                problems.append(f"request envelope is missing {missing} -> server answers -32602")
    elif "result" in msg or "error" in msg:
        kind = "response"
        if "result" in msg and "error" in msg:
            problems.append("a response carries result OR error, never both")
        if "result" in msg:
            if msg.get("id") is None:
                problems.append("a result response must echo the request id")
            if "resultType" not in msg["result"]:
                problems.append("result lacks resultType (a client treats it as 'complete')")
        if "error" in msg and not isinstance(msg["error"].get("code"), int):
            problems.append("error.code must be an integer")
    else:
        kind = "invalid"
        problems.append("neither method nor result/error present")
    return kind, problems


ENVELOPE = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
            "io.modelcontextprotocol/clientCapabilities": {},
            "io.modelcontextprotocol/clientInfo": {"name": "notebook", "version": "0.1"}}

samples = {
    "good request": {"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {"_meta": ENVELOPE}},
    "null id": {"jsonrpc": "2.0", "id": None, "method": "tools/list", "params": {"_meta": ENVELOPE}},
    "no envelope": {"jsonrpc": "2.0", "id": "a", "method": "tools/list"},
    "notification": {"jsonrpc": "2.0", "method": "notifications/progress",
                     "params": {"progressToken": "t", "progress": 1}},
    "old-style result": {"jsonrpc": "2.0", "id": 1, "result": {"tools": []}},
    "error": {"jsonrpc": "2.0", "id": 1, "error": {"code": -32601, "message": "Method not found"}},
}
for label, msg in samples.items():
    kind, problems = inspect_message(msg)
    print(f"{label:<17} -> {kind:<12} {problems or 'OK'}")

### 2. `_meta` 键规则与保留前缀
实现规范中的键语法：一个可选的、以 `/` 结尾的反向 DNS 前缀，后接一个名称。当前缀的<em>第二个</em>标签是 `modelcontextprotocol` 或 `mcp` 时，该前缀就是<strong>保留前缀（Reserved Prefix）</strong>。

In [ ]:
import re

LABEL = r"[A-Za-z](?:[A-Za-z0-9-]*[A-Za-z0-9])?"
NAME = r"(?:[A-Za-z0-9](?:[A-Za-z0-9._-]*[A-Za-z0-9])?)?"
KEY = re.compile(rf"^(?:(?P<prefix>{LABEL}(?:\.{LABEL})*)/)?(?P<name>{NAME})$")
OTEL_KEYS = {"traceparent", "tracestate", "baggage"}


def check_meta_key(key: str) -> str:
    m = KEY.match(key)
    if not m:
        return "INVALID"
    prefix = m.group("prefix")
    if prefix is None:
        return "OpenTelemetry exception (reserved, no prefix)" if key in OTEL_KEYS else "valid, unprefixed"
    labels = prefix.split(".")
    if len(labels) > 1 and labels[1] in ("modelcontextprotocol", "mcp"):
        return "RESERVED for MCP"
    return "valid, vendor prefix"


for key in ["io.modelcontextprotocol/protocolVersion", "dev.mcp/anything", "org.modelcontextprotocol.api/x",
            "com.mcp.tools/y", "com.example.mcp/tenant", "com.acme/tenant", "traceparent", "progressToken",
            "1bad.com/x", "com.acme/-name", "com.acme-/x"]:
    print(f"{key:<42} {check_meta_key(key)}")

注意 `com.example.mcp/tenant` <strong>不是</strong>保留的（它的第二个标签是 `example`），而 `com.mcp.tools/y` <strong>是</strong>保留的。

### 3. 一个错误码从何而来？
把分配策略写成代码，这样你一眼就能读懂任何错误码。

In [ ]:
MCP_DEFINED = {-32020: "HeaderMismatch", -32021: "MissingRequiredClientCapability",
               -32022: "UnsupportedProtocolVersion"}
JSONRPC_STANDARD = {-32700: "Parse error", -32600: "Invalid request", -32601: "Method not found",
                    -32602: "Invalid params", -32603: "Internal error"}


def explain_code(code: int) -> str:
    if code in JSONRPC_STANDARD:
        return f"JSON-RPC standard: {JSONRPC_STANDARD[code]}"
    if code in MCP_DEFINED:
        return f"MCP spec: {MCP_DEFINED[code]}"
    if -32019 <= code <= -32000:
        extra = " (old 'resource not found' - accept from legacy servers)" if code == -32002 else ""
        return "legacy implementation-defined range: assume nothing" + extra
    if -32099 <= code <= -32020:
        return "reserved for the MCP spec but undefined: a server MUST NOT emit it"
    if -32768 <= code <= -32000:
        return "JSON-RPC reserved range: do not use for app errors"
    return "application-defined (outside the JSON-RPC reserved range)"


for code in [-32700, -32602, -32020, -32021, -32022, -32002, -32001, -32042, -32050, -32500, -1, 4004]:
    print(f"{code:>7}  {explain_code(code)}")

### 4. 通过 stdio 从真实服务器触发每一种错误
现在，一个真实的 SDK 服务器（Server）作为子进程运行，我们用原始的 JSON 行与它对话。其中一个工具 `delete` 需要用户确认。它通过一个会发起征询（Elicitation）的解析器来声明这一点，因此服务器需要客户端具备 `elicitation` 能力。

In [ ]:
import queue
import subprocess
import sys
import tempfile
import textwrap
import threading
from pathlib import Path

workdir = Path(tempfile.mkdtemp())
(workdir / "lab_server.py").write_text(textwrap.dedent('''
    from typing import Annotated
    from pydantic import BaseModel
    from mcp.server import MCPServer
    from mcp.server.mcpserver import Elicit, Resolve

    mcp = MCPServer("error-lab", version="1.0.0")

    @mcp.tool()
    def add(a: int, b: int) -> int:
        """Add two integers."""
        return a + b

    @mcp.tool()
    def divide(a: float, b: float) -> float:
        """Divide a by b."""
        if b == 0:
            raise ValueError("division by zero")
        return a / b

    class Confirm(BaseModel):
        ok: bool

    def ask(target: str):
        return Elicit(f"Really delete {target}?", Confirm)

    @mcp.tool()
    def delete(target: str, confirm: Annotated[Confirm, Resolve(ask)]) -> str:
        """Delete something after the user confirms."""
        return f"deleted {target}" if confirm.ok else "kept"

    if __name__ == "__main__":
        mcp.run(transport="stdio")
'''))

proc = subprocess.Popen([sys.executable, str(workdir / "lab_server.py")], stdin=subprocess.PIPE,
                        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
inbox = queue.Queue()
threading.Thread(target=lambda: [inbox.put(line) for line in proc.stdout], daemon=True).start()


def send_raw(text: str, wait: float = 5.0):
    """Write one line to the server's stdin and return the next line it writes (or None)."""
    proc.stdin.write(text + "\n")
    proc.stdin.flush()
    try:
        return json.loads(inbox.get(timeout=wait))
    except queue.Empty:
        return None


def rpc(id_, method, params=None, *, meta=ENVELOPE):
    params = dict(params or {})
    if meta is not None:
        params["_meta"] = meta
    return send_raw(json.dumps({"jsonrpc": "2.0", "id": id_, "method": method, "params": params}))


reply = rpc(1, "server/discover")
show(reply)
print("classifier:", inspect_message(reply))

现在一次破坏一样东西，并把每个回复都交给两个辅助函数处理：

In [ ]:
def report(label, reply):
    if reply is None:
        print(f"{label:<34} -> (no reply)")
    elif "error" in reply:
        err = reply["error"]
        print(f"{label:<34} -> {err['code']} {explain_code(err['code'])}\n{'':<38}message: {err['message'][:90]}")
        if "data" in err:
            print(f"{'':<38}data: {json.dumps(err['data'])}")
    else:
        r = reply["result"]
        print(f"{label:<34} -> result resultType={r.get('resultType')!r} isError={r.get('isError')}")


report("unknown method", rpc(2, "tasks/list"))
report("missing _meta envelope", rpc(3, "tools/list", meta=None))
report("unsupported protocol version", rpc(4, "tools/list",
       meta={**ENVELOPE, "io.modelcontextprotocol/protocolVersion": "1900-01-01"}))
report("needs undeclared elicitation", rpc(5, "tools/call", {"name": "delete", "arguments": {"target": "prod"}}))
report("tool raises (NOT a protocol error)", rpc(6, "tools/call", {"name": "divide", "arguments": {"a": 1, "b": 0}}))

仔细看最后两个。在没有声明 `elicitation` 的情况下调用 `delete` 会得到 <strong>-32021</strong>，`data.requiredCapabilities` 会准确地告诉客户端需要添加什么。抛出异常的工具（Tool）<strong>不是</strong> JSON-RPC 错误：它是一个带有 `isError: true` 的 `complete` 结果，宿主会把它展示给模型。

如果客户端<em>确实</em>声明了 elicitation，会发生什么？服务器不能再发送自己的 `elicitation/create` 请求（Request）（在 2026-07-28 中服务器从不发起请求），所以它以 `resultType: "input_required"` 作答：

In [ ]:
reply = rpc(7, "tools/call", {"name": "delete", "arguments": {"target": "prod"}},
            meta={**ENVELOPE, "io.modelcontextprotocol/clientCapabilities": {"elicitation": {}}})
result = reply["result"]
print("resultType:", result["resultType"])
show(result["inputRequests"])
print("requestState (opaque, echoed back on retry):", result["requestState"][:40], "...")

第 5.3 课会完成这次往返。眼下的重点是：`resultType` 告诉客户端如何解析 `result`。

最后是格式错误的 JSON-RPC。规范说，解析错误要以 `id: null` 作答，但这个 SDK 的 stdio 服务器会直接丢弃它无法解析、或者不符合 JSON-RPC 结构的行。<strong>永远不要指望垃圾输入能得到回复。</strong> 一定要使用客户端超时。

In [ ]:
for label, line in [("not JSON at all", '{"jsonrpc": "2.0", "id": 8, "method": "tools/l'),
                    ("null id", json.dumps({"jsonrpc": "2.0", "id": None, "method": "tools/list",
                                            "params": {"_meta": ENVELOPE}})),
                    ("jsonrpc 1.0", json.dumps({"jsonrpc": "1.0", "id": 9, "method": "tools/list",
                                                "params": {"_meta": ENVELOPE}}))]:
    report(label, send_raw(line, wait=1.5))

report("server still healthy?", rpc(10, "tools/call", {"name": "add", "arguments": {"a": 2, "b": 40}}))
proc.stdin.close()
exit_code = proc.wait(timeout=10)   # closing stdin is the graceful shutdown signal

### 5. 通过 Streamable HTTP 的同样错误，以及状态码
在 HTTP 上，每个错误还会对应一个 HTTP 状态码：信封、请求头和版本问题对应 `400`，未知方法对应 `404`。这里服务器在 `127.0.0.1` 上的一个后台线程中运行。

In [ ]:
import logging
import socket
import time

import httpx
import uvicorn
from mcp.server import MCPServer

for noisy in ("httpx", "mcp", "uvicorn"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

http_server = MCPServer("error-lab-http")


@http_server.tool()
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


sock = socket.socket()
sock.bind(("127.0.0.1", 0))
port = sock.getsockname()[1]
sock.close()
uv = uvicorn.Server(uvicorn.Config(http_server.streamable_http_app(), host="127.0.0.1", port=port,
                                   log_level="warning"))
threading.Thread(target=uv.run, daemon=True).start()
while not uv.started:
    time.sleep(0.05)
URL = f"http://127.0.0.1:{port}/mcp"
web = httpx.Client(trust_env=False, timeout=10)
BASE_HEADERS = {"Accept": "application/json, text/event-stream", "Content-Type": "application/json",
                "MCP-Protocol-Version": "2026-07-28"}


def post(label, body, headers):
    r = web.post(URL, content=body if isinstance(body, str) else json.dumps(body),
                 headers={**BASE_HEADERS, **headers})
    data = r.json()
    code = data.get("error", {}).get("code")
    summary = f"error {code}: {data['error']['message'][:70]}" if code else f"result {data['result'].get('structuredContent')}"
    print(f"{label:<28} HTTP {r.status_code}  {summary}")


call = {"jsonrpc": "2.0", "id": 1, "method": "tools/call",
        "params": {"name": "add", "arguments": {"a": 2, "b": 3}, "_meta": ENVELOPE}}
post("well-formed", call, {"Mcp-Method": "tools/call", "Mcp-Name": "add"})
post("Mcp-Name disagrees", call, {"Mcp-Method": "tools/call", "Mcp-Name": "sub"})
post("unknown method", {"jsonrpc": "2.0", "id": 2, "method": "nope/x", "params": {"_meta": ENVELOPE}},
     {"Mcp-Method": "nope/x"})
post("unsupported version", {**call, "params": {**call["params"], "_meta": {
    **ENVELOPE, "io.modelcontextprotocol/protocolVersion": "1900-01-01"}}},
     {"Mcp-Method": "tools/call", "Mcp-Name": "add", "MCP-Protocol-Version": "1900-01-01"})
post("missing envelope", {"jsonrpc": "2.0", "id": 3, "method": "tools/list", "params": {}},
     {"Mcp-Method": "tools/list"})
post("not JSON", '{"jsonrpc": "2.0", "id": 4,', {"Mcp-Method": "tools/list"})

在 HTTP 上，解析错误<em>确实</em>会得到回复：`-32700`，`id: null`，因为服务器无法读取 id。第 2.3 课会深入讲解这些请求头。

### 6. `_meta` 透传：链路追踪上下文与厂商键
`Client.call_tool(..., meta=...)` 会把你的键合并进 SDK 构建的信封（Per-request Envelope）中。这个工具会原样回显收到的内容，让你看到 W3C 的 `traceparent` 和一个带厂商前缀的租户键与协议字段一起传递。

In [ ]:
from mcp import Client
from mcp.server.mcpserver import Context

echo = MCPServer("meta-echo")


@echo.tool()
def show_meta(ctx: Context) -> str:
    """Return the request's _meta exactly as the server received it."""
    return json.dumps(dict(ctx.request_context.meta or {}), indent=2, default=str)


async with Client(echo) as client:
    result = await client.call_tool("show_meta", {}, meta={
        "traceparent": "00-0af7651916cd43dd8448eb211c80319c-00f067aa0ba902b7-01",
        "com.acme/tenant": "tenant-42",
    })
    print(result.content[0].text)
    print("\nserverInfo in the result's _meta:", result.meta["io.modelcontextprotocol/serverInfo"])

SDK 为你填上了 `protocolVersion`、`clientCapabilities` 和 `clientInfo`。在服务器端，SDK 的 OpenTelemetry 中间件会从同一个 `_meta` 中读取 `traceparent`，用来确定其 span 的父级。

### 7. 图标，以及像对待不可信输入一样检查它们
服务器可以在自己的实现信息和工具上附加 `icons`。客户端（Client）在渲染任何东西之前都必须先审查它们。

In [ ]:
from mcp_types import Icon

PNG_DOT = "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8z8BQDwAEhQGAhKmMIQAAAABJRU5ErkJggg=="
iconic = MCPServer("iconic", icons=[Icon(src=PNG_DOT, mime_type="image/png", sizes=["1x1"])])


@iconic.tool(icons=[Icon(src="javascript:alert(1)"), Icon(src="https://iconic.example/calc.svg",
                                                            mime_type="image/svg+xml", sizes=["any"])])
def calc(expression: str) -> str:
    """Pretend calculator."""
    return expression


def icon_verdict(icon) -> str:
    scheme = icon.src.split(":", 1)[0].lower()
    if scheme not in ("https", "data"):
        return f"REJECT unsafe scheme {scheme!r}"
    if icon.mime_type == "image/svg+xml":
        return "allow only after sanitising the SVG (it can embed script)"
    return "ok: fetch without credentials, then verify magic bytes"


async with Client(iconic) as client:
    print("server icons:", [icon_verdict(i) for i in client.server_info.icons or []])
    for tool in (await client.list_tools()).tools:
        for icon in tool.icons or []:
            print(f"tool {tool.name}: {icon.src[:40]:<40} -> {icon_verdict(icon)}")

### 动手练习
1. 扩展 `inspect_message`，让它能标记出 `_meta` 中含有未通过 `check_meta_key` 的键的请求（Request），以及带有 `id: null` 的通知（Notification）。在几条手工构造的消息上重新运行它。
2. 在第 4 节中，再次调用 `delete`，但只声明 `{"elicitation": {"url": {}}}`。对于一个表单问题，服务器是否接受仅支持 URL 的征询？读一读 `data.requiredCapabilities` 来找出答案。
3. 在第 5 节中，发送一个格式正确的请求，请求头中写 `MCP-Protocol-Version: 2026-07-28`，而消息体的 `_meta` 中写 `"2025-11-25"`。哪个错误码胜出，`-32020` 还是 `-32022`？为什么先检查请求头是合理的？
4. 使用 `explain_code`，决定你自己的服务器在工具内部遇到“客户不存在”时应该使用哪个错误码。（陷阱题：它到底应不应该是一个 JSON-RPC 错误？）